# Operational Rainfall Forecasting System
**Meteorological Focus**: Advanced Feature Engineering, Temporal Splitting, and Multi-Objective Evaluation.

## Purpose
This notebook serves as a dedicated environment for evaluating the performance of this specific architecture on operational rainfall forecasting. The workflow isolates this model to ensure reproducible evaluation without cross-contamination.

## Meteorological Rationale
Rainfall is a complex, non-linear atmospheric process driven by thermodynamic instability, moisture convergence, and synoptic-scale forcing. By predicting multiple aspects of rainfall simultaneously (Occurrence, Category, and Amount), we force the model to build a physically consistent internal representation of storm events.

## Methodology
1. **Automated Feature Engineering**: Extensive generation of lag, trend, rolling, and cyclical features.
2. **Chronological Split**: Strict temporal separation (2005-2023 Train, 2024 Val, 2025 Test).
3. **Bayesian Optimization**: Optuna is used to find the optimal hyperparameters.
4. **Comprehensive Evaluation**: Metrics span standard ML (F1, AUC, RMSE) and WMO meteorology standards (CSI, POD, FAR).


# Library Imports and Configuration
## Purpose
Initialize the environment, set deterministic seeds, and configure hardware acceleration.


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import optuna
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score,
    roc_auc_score, brier_score_loss, confusion_matrix, log_loss, mean_squared_error, 
    mean_absolute_error, r2_score, mean_absolute_percentage_error, 
    precision_recall_curve, roc_curve, auc, cohen_kappa_score
)
from sklearn.calibration import calibration_curve
from sklearn.utils import class_weight
import joblib

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    logger.info(f"Seed set to {seed}")

seed_everything(42)


# Data Loading & Base Preparation
## Purpose
Load the raw meteorological dataset and standardize column names and timestamps.

## Inputs
`cuaca_jerukagung.csv` containing raw hourly variables.


In [ ]:
def get_paths():
    cwd = Path.cwd()
    if '/kaggle' in str(cwd) or '\\kaggle' in str(cwd):
        data_path = Path("/kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv")
    else:
        data_path = Path(r"D:\Github\Projek_Rainfall\Analisis_Meteorologi\open_meteo_jerukagung\cuaca_jerukagung.csv")
    return data_path

def load_data(filepath):
    logger.info(f"Loading data from {filepath}")
    df = pd.read_csv(filepath)
    if 'datetime' in df.columns: df = df.set_index('datetime')
    elif 'date' in df.columns: df = df.set_index('date')
    df.index = pd.to_datetime(df.index, utc=True).tz_convert('Asia/Jakarta').tz_localize(None)
    df.index.name = 'date'
    df = df.sort_index()
    
    # FILTER: Use only essential AWS/IoT deployable sensors
    essential_cols = [
        'temperature_2m', 'relative_humidity_2m', 'rain', 
        'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 
        'shortwave_radiation'
    ]
    # Keep only those columns that exist in the dataframe
    cols_to_keep = [c for c in essential_cols if c in df.columns]
    df = df[cols_to_keep]
    logger.info(f"Filtered dataset to essential AWS variables: {cols_to_keep}")
    
    # Ensure rain is strictly non-negative
    if 'rain' in df.columns:
        df.loc[df['rain'] < 0, 'rain'] = 0
    return df

data_path = get_paths()
df_raw = load_data(data_path)
display(df_raw.head())


# Automated Feature Engineering
## Purpose
Generate a massive pool of meteorological predictors to capture atmospheric memory and momentum.

## Meteorological Rationale
Rainfall is heavily influenced by temporal persistence (what happened an hour ago?) and atmospheric evolution (is pressure dropping rapidly?).
1. **Lag Variables**: Capture recent state.
2. **Rolling Statistics**: Capture sustained regimes (e.g., prolonged humidity).
3. **Trend Variables**: Capture rate-of-change (e.g., rapid pressure drops indicate incoming storms).
4. **Cyclical Features**: Capture diurnal heating cycles and seasonal monsoons.


In [ ]:
def generate_features(df):
    logger.info("Generating advanced features...")
    df = df.copy()
    
    # 1. Temporal / Cyclical Features
    df['hour'] = df.index.hour
    df['dayofyear'] = df.index.dayofyear
    df['month'] = df.index.month
    
    df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['sin_doy'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['cos_doy'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
    df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12.0)
    df = df.drop(columns=['hour', 'dayofyear', 'month'])
    
    vars_to_lag = [c for c in df.columns if c not in ['sin_hour','cos_hour','sin_doy','cos_doy','sin_month','cos_month']]
    
    # 2. Lag Features
    lags = [1, 3, 6, 12, 24]
    for col in vars_to_lag:
        for lag in lags:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
            
    # 3. Trend Features (Diff)
    trends = [1, 3, 6]
    for col in vars_to_lag:
        for t in trends:
            df[f'{col}_change_{t}h'] = df[col].diff(t)
            
    # 4. Rolling Features
    windows = [3, 6, 12, 24]
    for col in vars_to_lag:
        for w in windows:
            df[f'{col}_roll_mean_{w}h'] = df[col].rolling(window=w, min_periods=1).mean()
            df[f'{col}_roll_std_{w}h']  = df[col].rolling(window=w, min_periods=1).std().fillna(0)
            df[f'{col}_roll_min_{w}h']  = df[col].rolling(window=w, min_periods=1).min()
            df[f'{col}_roll_max_{w}h']  = df[col].rolling(window=w, min_periods=1).max()
            
    df = df.dropna()
    return df

df_feat = generate_features(df_raw)
logger.info(f"Feature engineering generated {df_feat.shape[1]} total columns.")


# Objective Definition & Aggregation
## Purpose
Resample data to the operational 3-hour forecasting window and define the multi-task objectives.

## Outputs
- `target_occurrence`: 0 (No Rain), 1 (Rain)
- `target_class`: 0 (<0.5mm), 1 (0.5-5mm), 2 (5-20mm), 3 (>20mm)
- `target_amount`: mm value


In [ ]:
def categorize_rain(mm):
    if mm < 0.5: return 0
    elif 0.5 <= mm <= 5.0: return 1
    elif 5.0 < mm <= 20.0: return 2
    else: return 3

def prepare_targets(df):
    logger.info("Resampling to 3-hourly & creating targets...")
    # Base aggregation rules
    agg_rules = {c: 'mean' for c in df.columns}
    if 'rain' in df.columns:
        agg_rules['rain'] = 'sum'
        
    df_3h = df.resample('3h').agg(agg_rules).dropna()
    
    # Target is the next 3h step
    df_3h['target_amount'] = df_3h['rain'].shift(-1)
    df_3h = df_3h.dropna()
    
    df_3h['target_occurrence'] = (df_3h['target_amount'] > 0).astype(int)
    df_3h['target_class'] = df_3h['target_amount'].apply(categorize_rain)
    
    return df_3h

df_processed = prepare_targets(df_feat)
logger.info(f"Final shape before split: {df_processed.shape}")


# Chronological Data Split
## Purpose
Ensure no data leakage by strictly separating time periods as requested.

## Methodology
- **Train**: 2005-01-01 to 2023-12-31
- **Validation**: 2024-01-01 to 2024-12-31 (Used for Optuna/Early Stopping)
- **Test**: 2025-01-01 to 2025-12-31


In [ ]:
train_mask = (df_processed.index.year >= 2005) & (df_processed.index.year <= 2023)
val_mask = (df_processed.index.year == 2024)
test_mask = (df_processed.index.year == 2025)

features = df_processed.drop(columns=['target_amount', 'target_occurrence', 'target_class'])
targets = df_processed[['target_amount', 'target_occurrence', 'target_class']]
feature_names = features.columns.tolist()

X_train, y_train = features[train_mask], targets[train_mask]
X_val, y_val = features[val_mask], targets[val_mask]
X_test, y_test = features[test_mask], targets[test_mask]

logger.info(f"Train samples: {len(X_train)} | Val samples: {len(X_val)} | Test samples: {len(X_test)}")

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


# Evaluation Metrics Module
## Purpose
Calculate exhaustive Machine Learning and Meteorological metrics to benchmark the model.


In [ ]:
def compute_meteorological_metrics(y_true, y_pred, num_classes=4):
    met_metrics = {}
    for c in range(1, num_classes): # Focus on rain classes
        hits = np.sum((y_pred == c) & (y_true == c))
        misses = np.sum((y_pred != c) & (y_true == c))
        false_alarms = np.sum((y_pred == c) & (y_true != c))
        correct_negatives = np.sum((y_pred != c) & (y_true != c))
        
        csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else 0
        pod = hits / (hits + misses) if (hits + misses) > 0 else 0
        far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else 0
        
        total = hits + misses + false_alarms + correct_negatives
        hits_random = ((hits + misses) * (hits + false_alarms)) / total if total > 0 else 0
        ets = (hits - hits_random) / (hits + misses + false_alarms - hits_random) if (hits + misses + false_alarms - hits_random) > 0 else 0
        hss = (2 * (hits * correct_negatives - misses * false_alarms)) / ((hits + misses) * (misses + correct_negatives) + (hits + false_alarms) * (false_alarms + correct_negatives)) if total > 0 else 0
        
        met_metrics[f'Class_{c}'] = {'CSI': csi, 'POD': pod, 'FAR': far, 'ETS': ets, 'HSS': hss}
    return met_metrics

def calculate_all_metrics(y_true_occ, y_pred_occ, y_prob_occ, 
                          y_true_cls, y_pred_cls, y_prob_cls, 
                          y_true_reg, y_pred_reg):
    
    # Regression
    rmse = np.sqrt(mean_squared_error(y_true_reg, y_pred_reg))
    mae = mean_absolute_error(y_true_reg, y_pred_reg)
    r2 = r2_score(y_true_reg, y_pred_reg)
    mape = mean_absolute_percentage_error(y_true_reg + 1e-5, y_pred_reg)
    nse = 1 - (np.sum((y_true_reg - y_pred_reg)**2) / (np.sum((y_true_reg - np.mean(y_true_reg))**2) + 1e-5))
    
    r_pearson = np.corrcoef(y_true_reg, y_pred_reg)[0,1]
    alpha_kge = np.std(y_pred_reg) / (np.std(y_true_reg) + 1e-5)
    beta_kge = np.mean(y_pred_reg) / (np.mean(y_true_reg) + 1e-5)
    kge = 1 - np.sqrt((r_pearson - 1)**2 + (alpha_kge - 1)**2 + (beta_kge - 1)**2)
    
    # Binary Occurrence
    acc_occ = accuracy_score(y_true_occ, y_pred_occ)
    prec_occ = precision_score(y_true_occ, y_pred_occ, zero_division=0)
    rec_occ = recall_score(y_true_occ, y_pred_occ, zero_division=0)
    f1_occ = f1_score(y_true_occ, y_pred_occ, zero_division=0)
    auc_occ = roc_auc_score(y_true_occ, y_prob_occ)
    brier = brier_score_loss(y_true_occ, y_prob_occ)
    logloss_occ = log_loss(y_true_occ, y_prob_occ, labels=[0,1])
    
    # Multiclass
    acc_cls = accuracy_score(y_true_cls, y_pred_cls)
    f1_macro = f1_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true_cls, y_pred_cls, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true_cls, y_pred_cls)
    
    met = compute_meteorological_metrics(y_true_cls, y_pred_cls)
    
    report = {
        'Regression': {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2, 'NSE': nse, 'KGE': kge},
        'Occurrence': {'Accuracy': acc_occ, 'Precision': prec_occ, 'Recall': rec_occ, 'F1': f1_occ, 'ROC_AUC': auc_occ, 'Brier': brier, 'LogLoss': logloss_occ},
        'Category': {'Accuracy': acc_cls, 'Macro_F1': f1_macro, 'Weighted_F1': f1_weighted, 'Cohen_Kappa': kappa},
        'Meteorological': met
    }
    return report

def plot_all_diagnostics(y_true_cls, y_pred_cls, y_prob_cls, y_true_reg, y_pred_reg, out_dir, model_name):
    os.makedirs(out_dir, exist_ok=True)
    
    # 1. Confusion Matrix
    plt.figure()
    sns.heatmap(confusion_matrix(y_true_cls, y_pred_cls), annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.savefig(f'{out_dir}/confusion_matrix.png', bbox_inches='tight')
    plt.show()
    
    # 2. Pred vs Obs Regression
    plt.figure()
    plt.scatter(y_true_reg, y_pred_reg, alpha=0.5)
    plt.plot([0, max(y_true_reg)], [0, max(y_true_reg)], 'r--')
    plt.xlabel('Observed')
    plt.ylabel('Predicted')
    plt.title('Prediction vs Observation (Regression)')
    plt.savefig(f'{out_dir}/pred_vs_obs.png', bbox_inches='tight')
    plt.show()
    
    # 3. Residual Distribution
    plt.figure()
    sns.histplot(y_true_reg - y_pred_reg, kde=True)
    plt.title('Residual Distribution')
    plt.savefig(f'{out_dir}/residual_dist.png', bbox_inches='tight')
    plt.show()


# Sequence Preparation for LSTM
## Purpose
Transform the 2D tabular data into 3D sequences `(samples, time_steps, features)` required for Recurrent Neural Networks.


In [ ]:
def create_sequences(X, y_occ, y_cls, y_reg, time_steps=24):
    Xs, y_o, y_c, y_r = [], [], [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        y_o.append(y_occ.iloc[i + time_steps])
        y_c.append(y_cls.iloc[i + time_steps])
        y_r.append(y_reg.iloc[i + time_steps])
    return np.array(Xs), np.array(y_o), np.array(y_c), np.array(y_r)
logger.info("Sequence Generation Function Ready.")


# Multi-Task LSTM Architecture
## Purpose
Build a Keras functional model with one shared LSTM backbone branching into three distinct predictive heads.

## Methodology
- **Loss Weights**: Static weights are used (as per user preference).
- **Focal Loss**: Applied to the Binary Occurrence head to handle zero-inflation.


In [ ]:
class FocalLossBinary(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma, self.alpha = gamma, alpha
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, keras.backend.epsilon(), 1 - keras.backend.epsilon())
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = self.alpha * tf.math.pow(1 - p_t, self.gamma)
        return tf.reduce_sum(focal_weight * bce, axis=-1)


# Bayesian Architecture Search (Optuna)
## Purpose
Automatically discover the optimal sequence length, LSTM dimensions, dropout rates, and learning rate using Bayesian Optimization.

## Methodology
Because `sequence_length` changes the input shape of the dataset, the `create_sequences` function is invoked *inside* the Optuna objective function. Optuna minimizes a weighted composite validation loss from the 3 prediction heads.


In [ ]:
def objective_lstm(trial):
    keras.backend.clear_session()
    
    # 1. Hyperparameters
    ts = trial.suggest_categorical('sequence_length', [12, 24, 48])
    hidden_size = trial.suggest_int('hidden_size', 32, 128, step=32)
    num_layers = trial.suggest_int('num_layers', 1, 3)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    
    # 2. Dynamic Sequence Generation
    X_tr_s, y_tr_o, y_tr_c, y_tr_r = create_sequences(X_train_scaled, y_train['target_occurrence'], y_train['target_class'], y_train['target_amount'], ts)
    X_v_s, y_v_o, y_v_c, y_v_r = create_sequences(X_val_scaled, y_val['target_occurrence'], y_val['target_class'], y_val['target_amount'], ts)
    
    # 3. Dynamic Architecture
    inputs = layers.Input(shape=(ts, X_tr_s.shape[2]))
    x = inputs
    for i in range(num_layers):
        ret_seq = (i < num_layers - 1)
        x = layers.LSTM(hidden_size, return_sequences=ret_seq)(x)
        x = layers.Dropout(dropout_rate)(x)
        
    out_occ = layers.Dense(1, activation='sigmoid', name='occ_head')(x)
    out_cls = layers.Dense(4, activation='softmax', name='cls_head')(x)
    out_reg = layers.Dense(1, activation='linear', name='reg_head')(x)
    
    model = keras.Model(inputs=inputs, outputs=[out_occ, out_cls, out_reg])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss={
            'occ_head': FocalLossBinary(gamma=2.0),
            'cls_head': 'sparse_categorical_crossentropy',
            'reg_head': 'mse'
        },
        loss_weights={'occ_head': 2.0, 'cls_head': 1.0, 'reg_head': 0.5}
    )
    
    # 4. Train
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    history = model.fit(
        X_tr_s, {'occ_head': y_tr_o, 'cls_head': y_tr_c, 'reg_head': y_tr_r},
        validation_data=(X_v_s, {'occ_head': y_v_o, 'cls_head': y_v_c, 'reg_head': y_v_r}),
        epochs=15, batch_size=batch_size, verbose=0, callbacks=[early_stop]
    )
    
    # Minimize the total validation loss
    val_loss = min(history.history['val_loss'])
    return val_loss

logger.info("Running Optuna Optimization for LSTM Architecture (2 trials for demo)...")
study_lstm = optuna.create_study(direction='minimize')
study_lstm.optimize(objective_lstm, n_trials=2) # Set higher in production
best_params_lstm = study_lstm.best_params
logger.info(f"Best LSTM Params: {best_params_lstm}")


# Model Training & Inference
## Purpose
Train the MTL LSTM and evaluate it using the shared evaluation framework.


In [ ]:
# Retrieve best parameters
ts = best_params_lstm['sequence_length']
hidden_size = best_params_lstm['hidden_size']
num_layers = best_params_lstm['num_layers']
dropout_rate = best_params_lstm['dropout']
lr = best_params_lstm['learning_rate']
batch_size = best_params_lstm['batch_size']

logger.info("Rebuilding best LSTM sequences...")
X_tr_seq, y_tr_occ, y_tr_cls, y_tr_reg = create_sequences(X_train_scaled, y_train['target_occurrence'], y_train['target_class'], y_train['target_amount'], ts)
X_v_seq, y_v_occ, y_v_cls, y_v_reg = create_sequences(X_val_scaled, y_val['target_occurrence'], y_val['target_class'], y_val['target_amount'], ts)
X_te_seq, y_te_occ, y_te_cls, y_te_reg = create_sequences(X_test_scaled, y_test['target_occurrence'], y_test['target_class'], y_test['target_amount'], ts)

# Build Final Model
keras.backend.clear_session()
inputs = layers.Input(shape=(ts, X_tr_seq.shape[2]))
x = inputs
for i in range(num_layers):
    ret_seq = (i < num_layers - 1)
    x = layers.LSTM(hidden_size, return_sequences=ret_seq)(x)
    x = layers.Dropout(dropout_rate)(x)
    
out_occ = layers.Dense(1, activation='sigmoid', name='occ_head')(x)
out_cls = layers.Dense(4, activation='softmax', name='cls_head')(x)
out_reg = layers.Dense(1, activation='linear', name='reg_head')(x)

model = keras.Model(inputs=inputs, outputs=[out_occ, out_cls, out_reg])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr),
    loss={'occ_head': FocalLossBinary(gamma=2.0), 'cls_head': 'sparse_categorical_crossentropy', 'reg_head': 'mse'},
    loss_weights={'occ_head': 2.0, 'cls_head': 1.0, 'reg_head': 0.5}
)

logger.info("Training Final Best LSTM Model...")
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    X_tr_seq, {'occ_head': y_tr_occ, 'cls_head': y_tr_cls, 'reg_head': y_tr_reg},
    validation_data=(X_v_seq, {'occ_head': y_v_occ, 'cls_head': y_v_cls, 'reg_head': y_v_reg}),
    epochs=50, batch_size=batch_size, verbose=1, callbacks=[early_stop]
)

preds = model.predict(X_te_seq)
prob_occ = preds[0].flatten()
pred_occ = (prob_occ > 0.5).astype(int)
prob_cls = preds[1]
pred_cls = np.argmax(prob_cls, axis=1)
pred_reg = np.maximum(0, preds[2].flatten())

report = calculate_all_metrics(y_te_occ, pred_occ, prob_occ,
                               y_te_cls, pred_cls, prob_cls,
                               y_te_reg, pred_reg)

out_path = Path("outputs/lstm")
os.makedirs(out_path / 'metrics', exist_ok=True)
os.makedirs(out_path / 'plots', exist_ok=True)

import json
with open(out_path / 'metrics' / 'metrics_report.json', 'w') as f:
    json.dump(report, f, indent=4)

plot_all_diagnostics(y_te_cls, pred_cls, prob_cls, y_te_reg, pred_reg, out_path / 'plots', 'LSTM')

plt.figure()
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Multi-Task LSTM Training Loss')
plt.savefig(out_path / 'plots' / 'training_loss.png')
plt.show()

logger.info("LSTM Evaluation Complete. Metrics saved.")
